In [1]:
import pandas as pd
import time
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# kullanilicak modellerin aktarilmasi
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier



In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# scaler'i tanimla
scaler = StandardScaler()

# egitim verisini olceklendirme
X_train_scaled = scaler.fit_transform(X_train)

# sadece transform yapilir cunku egitimdeki oranlarin kullanilmasini istiyoruz (test verisini kopyalamamali)
test_df_scaled = scaler.transform(test_df)

# veriyi egitim ve dogrulama olarak ayir (X_train_scaled ile)
X_egitim, X_val, y_egitim, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
modeller = {
    "Logistic Regression": LogisticRegression(solver='liblinear',max_iter=1000,random_state=42),
    "Naive Bayes": GaussianNB(),

    # max_depth ile agacın derinligini kisitliyoruz (buduyoruz)
    # min_samples_split ile bir dalin ayrilmasi icin en az 10 kisi olmasini sart kosuyoruz
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42),

    # eski GridSearch calismasinda buldugumuz sampiyon parametreleri buraya entegre ettik
    "XGBoost": XGBClassifier(max_depth=4, learning_rate=0.1, n_estimators=100, random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(max_depth=4, learning_rate=0.1, n_estimators=100, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6, random_state=42, verbose=False)

}

sonuclar = []

print(f"{'Model adi':<25} | {'Sure (sn)':<10} | {'Dogruluk':<10} | {'F1 skoru'}")
print("-" * 65)

for isim, model in modeller.items():
    baslangic = time.time()

    model.fit(X_train, y_train)
    tahmin = model.predict(X_val)

    sure = time.time() - baslangic
    acc = accuracy_score(y_val, tahmin)
    f1 = f1_score(y_val, tahmin)

    sonuclar.append({"Model": isim, "Dogruluk": acc, "F1 Skoru": f1})
    print(f"{isim:<25} | {sure:<10.3f} | {acc:<10.3f} | {f1:.3f}")

# sonuclari grafige dokme
sonuclar_df = pd.DataFrame(sonuclar).sort_values(by="Dogruluk", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x="Dogruluk", y="Model", data=sonuclar_df, palette="viridis")
plt.title("Modellerin Dogruluk (Accuracy) Karsilastirmasi", fontsize=14, fontweight='bold')
plt.xlabel("Dogruluk Skoru", fontsize=12)
plt.ylabel("Algoritmalar", fontsize=12)
plt.xlim(0.65, 0.85) # farkı daha net gormek icin x eksenini daraltiyoruz
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

NameError: name 'X_train' is not defined

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. X (ozellikler) ve y (hedef) degiskenlerini ayirma islemi
X = train_df.drop('Transported', axis=1)
y = train_df['Transported'].astype(int) # true/false degerlerini 1 ve 0'a ceviriyoruz

# veriyi egitim ve dogrulama olarak bolme
X_egitim, X_val, y_egitim, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# olceklendirme islemi
scaler = StandardScaler()

# veri sizintisini onlemek için fit_transform'u sadece egitim verisine uyguluyoruz
X_egitim_scaled = scaler.fit_transform(X_egitim)

# validation verisine ise sadece egitim verisinden ogrenilen kurallar uygulanır
X_val_scaled = scaler.transform(X_val)

print("Veriler bsşariyla bolundu ve sinir agları icin olceklendirildi.")

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import time

print("Yapay sinir agi (ANN) egitiliyor...")
baslangic = time.time()

# model tanimlanir
# hidden_layer_sizes=(100, 50): ilk katmanda 100, ikinci katmanda 50 noron (hucre) var
ann_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500, # maksimum deneme sayisi
    activation='relu', # sinir aglarının en sevdiği aktivasyon fonksiyonu
    solver='adam', # agirliklari guncelleyen motor
    random_state=42
)

ann_model.fit(X_egitim_scaled, y_egitim)

# validation setinde test ediyoruz
ann_tahmin = ann_model.predict(X_val_scaled)
ann_acc = accuracy_score(y_val, ann_tahmin)

sure = time.time() - baslangic

print("-" * 50)
print(f"Egitim tamamlandi. Sure: {sure:.2f} saniye")
print(f"Yapay sinir agi (ANN) lokal dogruluk skoru: {ann_acc:.4f}")
print("-" * 50)

In [ ]:
from transformers import pipeline

print("Hugging face LLM siniflandirici yukleniyor...\n")

# facebook/bart-large-mnli modeli metinleri okuyup verdigimiz etiketlere gore siniflandirmada uzmandir
llm_tahminci = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# verimizi LLM'in anlayacagi bir hikayeye ceviriyoruz
yolcu_profili = "Bu yolcu Europa gezegeninden geliyor, zengin biri, VIP güvertesinde seyahat ediyor. Ancak kriyojenik uykuya yatmamış ve gemide çok fazla para harcamış."

# LLM'in secmesi gereken etiketler (Transported: true veya false durumu)
olasi_sonuclar = [
    "Bu yolcu uzay-zaman anomalisine kapılıp taşınmıştır.",
    "Bu yolcu gemide kalmış ve taşınmamıştır."
]

# modele soruyoruz
print("🕵️ LLM dusunuyor...\n")
sonuc = llm_tahminci(yolcu_profili, candidate_labels=olasi_sonuclar)

# Sonuçları yazdırma
print(f"📖 Yolcu ozeti: {yolcu_profili}")
print("-" * 50)
print(f"🥇 LLM'in en guclu tahmini: {sonuc['labels'][0]}")
print(f"🎯 Eminlik oranı: % {sonuc['scores'][0] * 100:.2f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

print("Kör nokta analizi basliyor...\n")

# modelleri hazirlama
print("Modeller analiz icin hizlica egitiliyor...")
model_cat = CatBoostClassifier(random_state=42, verbose=False)
model_lgb = LGBMClassifier(random_state=42, verbose=-1)
model_xgb = XGBClassifier(random_state=42, eval_metric='logloss')

model_cat.fit(X_egitim_scaled, y_egitim)
model_lgb.fit(X_egitim_scaled, y_egitim)
model_xgb.fit(X_egitim_scaled, y_egitim)

# tahminleri alma ve tablo yapma
tahmin_cat = model_cat.predict(X_val_scaled)
tahmin_lgb = model_lgb.predict(X_val_scaled)
tahmin_xgb = model_xgb.predict(X_val_scaled)

# yorumlamak icin olceklenmis orijinal X_val verisi kopyalanir
analiz_df = X_val.copy()
analiz_df['Gercek_Sonuc'] = y_val.values

# modellerin tahminleri eklenir
analiz_df['Cat_Tahmin'] = tahmin_cat
analiz_df['Lgb_Tahmin'] = tahmin_lgb
analiz_df['Xgb_Tahmin'] = tahmin_xgb

# dogruluk kontrolu (1=Dogru, 0=Yanlis)
analiz_df['Cat_Dogru'] = (analiz_df['Cat_Tahmin'] == analiz_df['Gercek_Sonuc']).astype(int)
analiz_df['Lgb_Dogru'] = (analiz_df['Lgb_Tahmin'] == analiz_df['Gercek_Sonuc']).astype(int)
analiz_df['Xgb_Dogru'] = (analiz_df['Xgb_Tahmin'] == analiz_df['Gercek_Sonuc']).astype(int)

# her satirda kac model uzlasti skoru
analiz_df['Konsey_Skoru'] = analiz_df['Cat_Dogru'] + analiz_df['Lgb_Dogru'] + analiz_df['Xgb_Dogru']

# veriyi 3 ana bolgeye ayirma
aydinlik_bolge = analiz_df[analiz_df['Konsey_Skoru'] == 3] # 3 dev model de %100 emin ve dogru
gri_bolge = analiz_df[(analiz_df['Konsey_Skoru'] == 1) | (analiz_df['Konsey_Skoru'] == 2)] # modeller anlasamiyor
karanlik_bolge = analiz_df[analiz_df['Konsey_Skoru'] == 0] # 3 model de kesinlikle yanlis bildi

print(f"Aydinlik bolge (Ucu de dogru): {len(aydinlik_bolge)} yolcu")
print(f"Gri Bolge (Kararsizlar): {len(gri_bolge)} yolcu")
print(f"Karanlık Bolge (Ucu de Yanlis): {len(karanlik_bolge)} yolcu\n")

# oruntu analizi
print("Modeller hangi profili anlayamiyor...\n")

kiyaslama_sutunlari = ['Age', 'Total_Expense', 'CryoSleep']

karsilastirma_tablosu = pd.DataFrame({
    "Basarili profil (Aydinlik)": aydinlik_bolge[kiyaslama_sutunlari].mean(),
    "Basarisiz profil (Karanlik)": karanlik_bolge[kiyaslama_sutunlari].mean()
}).round(2)

print(karsilastirma_tablosu)
print("-" * 50)

# basarisiz bolgenin gorsel analizi
if 'HomePlanet_Europa' in karanlik_bolge.columns:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.kdeplot(aydinlik_bolge['Total_Expense'], label='Aydinlik (Dogru)', fill=True, color='green', alpha=0.3)
    sns.kdeplot(karanlik_bolge['Total_Expense'], label='Karanlik (Yanlis)', fill=True, color='red', alpha=0.5)
    plt.title("Harcamalara gore kor nokta dağilimi")
    plt.xlim(0, 5000)
    plt.legend()

    plt.subplot(1, 2, 2)
    gezegen_toplamlari = karanlik_bolge[['HomePlanet_Europa', 'HomePlanet_Mars']].sum()
    gezegen_toplamlari['Earth_veya_Diger'] = len(karanlik_bolge) - gezegen_toplamlari.sum()
    gezegen_toplamlari.plot(kind='pie', autopct='%1.1f%%', colors=['#A9CCE3', '#F5B041', '#A3E4D7'])
    plt.title("Yanlis Bilinen Yolcularin Gezegeni")
    plt.ylabel('')

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# yeni method olarak StackingClassifier ve LogisticRegression uygulanmasi
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

print("Yeni method uygulamasi basliyor...\n")

# veriler ham halleriyle yuklenir
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# hedef degisken ayrilir ve test seti birlestirilir (islemleri tek seferde yapmak icin)
y = train_df['Transported'].astype(int)
train_df.drop('Transported', axis=1, inplace=True)

tum_veri = pd.concat([train_df, test_df], ignore_index=True)
harcama_sutunlari = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# PassengerId GGGG_PP formatinda "_" karakterinden bolunup ilk kisim (grup) alinir
tum_veri['Grup'] = tum_veri['PassengerId'].apply(lambda x: x.split('_')[0])

# her grupta kaç kisi oldugunu sayip yeni bir sutuna yaziyoruz
grup_sayilari = tum_veri['Grup'].value_counts().to_dict()
tum_veri['Grup_Boyutu'] = tum_veri['Grup'].map(grup_sayilari)

print("Grup_Boyutu basariyla cikarildi.")

# mantiksal doldurma
# once toplam harcama bulma islemi
tum_veri['Total_Expense'] = tum_veri[harcama_sutunlari].sum(axis=1)

# harcama 0'dan buyuksa kesinlikle uyumuyordur
tum_veri.loc[(tum_veri['Total_Expense'] > 0) & (tum_veri['CryoSleep'].isnull()), 'CryoSleep'] = False

# cryosleep=1 ise kesinlikle harcamasi 0'dir
for col in harcama_sutunlari:
    tum_veri.loc[(tum_veri['CryoSleep'] == True) & (tum_veri[col].isnull()), col] = 0

print("CryoSleep ve Harcama bosluklari mantikla dolduruldu.")

# cabin; deck, num, side olarak bolunur
tum_veri[['Deck', 'Num', 'Side']] = tum_veri['Cabin'].str.split('/', expand=True)

# kabin numarasi sayiya cevrilir
tum_veri['Num'] = pd.to_numeric(tum_veri['Num'], errors='coerce')

# gemi 300'luk bloklar halinde bolgelere ayrilir
tum_veri['Gemi_Bolgesi'] = pd.cut(tum_veri['Num'],
                                  bins=[-1, 300, 600, 900, 1200, 1500, 2000],
                                  labels=['Bolge_1', 'Bolge_2', 'Bolge_3', 'Bolge_4', 'Bolge_5', 'Bolge_6'])

print("Kabin numaralarindan Gemi_Bolgesi oluşturuldu.")

tum_veri['Expenditure_Type'] = pd.cut(tum_veri['Total_Expense'],
                                      bins=[-1, 1, 1000, 100000],
                                      labels=['No_Spend', 'Medium_Spend', 'High_Spend'])

tum_veri['VIP_Deck'] = tum_veri['Deck'].apply(lambda x: 1 if x in ['B', 'C'] else 0)

# veri temizligi ve sayisallastirma
# isi biten sutunlar cope atilir
dusulecek_sutunlar = ['PassengerId', 'Cabin', 'Name', 'Grup', 'Num']
tum_veri.drop(columns=dusulecek_sutunlar, inplace=True)

# kalan tum metinler sayisala cevrilir (one-hot encoding)
tum_veri = pd.get_dummies(tum_veri, drop_first=True)

# Eğitim ve test setini tekrar ayırıyoruz
# egitim ve test seti verisi tekrar ayirilir
X_train_full = tum_veri.iloc[:len(train_df)]
X_test_final = tum_veri.iloc[len(train_df):]

# modeli test etmek icin kendi icinde train/val olarak bolunur
X_egitim, X_val, y_egitim, y_val = train_test_split(X_train_full, y, test_size=0.2, random_state=42)

# stacking (kurmay zekasi) kurulumu
print("\nModeller egitiliyor ve stacking kuruluyor...")

# base estimators
cat = CatBoostClassifier(random_state=42, verbose=False)
lgb = LGBMClassifier(random_state=42, verbose=-1)
xgb = XGBClassifier(random_state=42, eval_metric='logloss')

# kurmay modelimiz (meta model - logistic regression)
# bu model, asagidaki 3 tahmini alip hangi modelin nerede hakli oldugunu soyleyecek
meta_model = LogisticRegression()

kurmay_sistemi = StackingClassifier(
    estimators=[('cat', cat), ('lgb', lgb), ('xgb', xgb)],
    final_estimator=meta_model,
    cv=5 # modellerin zafiyetlerini ogrenmek icin 5 capraz dogrulama yapar
)

# kurmay sistemini egitiyoruz
kurmay_sistemi.fit(X_egitim, y_egitim)

# final skorları
print("="*50)
print("Final skorlari")
print("="*50)
print(f"Eski voting skoru: ~0.80 - 0.81")
print(f"Yeni stacking skoru: {accuracy_score(y_val, kurmay_sistemi.predict(X_val)):.4f}")
print("="*50)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

print("Grafikler hazirlaniyor...")

plot_df = pd.read_csv("train.csv")

# grup boyutu tekrar cikarilir
plot_df['Grup'] = plot_df['PassengerId'].apply(lambda x: x.split('_')[0])
plot_df['Grup_Boyutu'] = plot_df['Grup'].map(plot_df['Grup'].value_counts().to_dict())

# gemi bolgesi tekrar cikarilir
plot_df[['Deck', 'Num', 'Side']] = plot_df['Cabin'].str.split('/', expand=True)
plot_df['Num'] = pd.to_numeric(plot_df['Num'], errors='coerce')
plot_df['Gemi_Bolgesi'] = pd.cut(plot_df['Num'],
                                  bins=[-1, 300, 600, 900, 1200, 1500, 2000],
                                  labels=['Bolge_1', 'Bolge_2', 'Bolge_3', 'Bolge_4', 'Bolge_5', 'Bolge_6'])

# grafik cizimi
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Final taktiklerinin analizi", fontsize=18, fontweight='bold')

# grafik 1: grup boyutu ve tasınma orani)
sns.barplot(data=plot_df, x='Grup_Boyutu', y='Transported', palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title("Grup boyutu ve tasinma ihtimali", fontsize=12)
axes[0, 0].set_ylabel("Tasinma orani (Transported %)")
axes[0, 0].set_xlabel("Gruptaki kisi sayisi")

# grafik 2: gemi topografyasi (kabin bolgesi)
sns.barplot(data=plot_df, x='Gemi_Bolgesi', y='Transported', palette='magma', ax=axes[0, 1])
axes[0, 1].set_title("Gemi bolgelerine gore anomalinin etkisi", fontsize=12)
axes[0, 1].set_ylabel("Tasinma orani")
axes[0, 1].set_xlabel("Kabin bolgesi (Onden arkaya)")

# grafik 3: modellerin karsilastirilmasi (Hafizadaki modellerin skorlari)
try:
    skorlar = {
        'CatBoost': accuracy_score(y_val, cat.predict(X_val)),
        'LightGBM': accuracy_score(y_val, lgb.predict(X_val)),
        'XGBoost': accuracy_score(y_val, xgb.predict(X_val)),
        'Stacking (Kurmay)': accuracy_score(y_val, kurmay_sistemi.predict(X_val))
    }

    sns.barplot(x=list(skorlar.keys()), y=list(skorlar.values()), palette=['#A9CCE3', '#A9CCE3', '#A9CCE3', '#E74C3C'], ax=axes[1, 0])
    axes[1, 0].set_title("Modellerin dogruluk (Accuracy) yarisi", fontsize=12)
    axes[1, 0].set_ylim(0.70, 0.85) # farki net gormek için alt siniri 0.70 yaptik

    # barların ustune sayilari yazdirma
    for i, v in enumerate(skorlar.values()):
        axes[1, 0].text(i, v + 0.002, f"{v:.4f}", ha='center', fontweight='bold')
except NameError:
    axes[1, 0].text(0.5, 0.5, "Modeller hafizada bulunamadi.\nLutfen once egitim hucresini calistirin.",
                    ha='center', va='center', fontsize=12)
    axes[1, 0].set_title("Modellerin kapismasi")

axes[1, 1].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
print("Final kaggle dosyasi hazirlaniyor...\n")

# modelin egitildigi sutun sirasiyla test verisinin sirasi birebir ayni olmali
X_test_final = X_test_final[X_egitim.columns]

# stacking ile tahmin: egittigimiz en guclu model ile test verisi uzerinde tahmin
final_tahminler = kurmay_sistemi.predict(X_test_final)

# kaggle formatina donusturme
final_tahminler_bool = [True if val == 1 else False for val in final_tahminler]

# submission dosyasini olusturma
orijinal_test = pd.read_csv("test.csv")

submission_df = pd.DataFrame({
    'PassengerId': orijinal_test['PassengerId'],
    'Transported': final_tahminler_bool
})

# csv olarak kaydetme
dosya_adi = "submission_endgame_stacking.csv"
submission_df.to_csv(dosya_adi, index=False)

print(f"{dosya_adi} dosyasi basariyla olusturuldu.")